In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Load training price data (pre-2024 only)
train = pd.read_parquet('../data/working/equities_train.parquet')
print(f"Training data: {len(train):,} rows, {train['ticker'].nunique()} tickers")
print(f"Date range: {train['date'].min().date()} to {train['date'].max().date()}")

# Load membership panel (to know who's in the index each month)
membership = pd.read_parquet('../data/raw/membership_panel.parquet')
membership['snapshot_date'] = pd.to_datetime(membership['snapshot_date'])
print(f"\nMembership: {len(membership):,} rows")

# Load benchmark
bench = pd.read_parquet('../data/working/benchmark_train.parquet')
bench['date'] = pd.to_datetime(bench['date'])
print(f"Benchmark: {len(bench):,} rows")

# Load SONIA
sonia = pd.read_parquet('../data/working/sonia_train.parquet')
sonia['date'] = pd.to_datetime(sonia['date'])
print(f"SONIA: {len(sonia):,} rows")

print("\nTraining columns:", list(train.columns))
print("\nSample:")
train.head()

Training data: 1,686,031 rows, 635 tickers
Date range: 2009-07-01 to 2023-12-29

Membership: 48,210 rows
Benchmark: 7,326 rows
SONIA: 3,663 rows

Training columns: ['ticker', 'date', 'CUR_MKT_CAP', 'EQY_SH_OUT', 'PX_HIGH', 'PX_LAST', 'PX_LOW', 'PX_OPEN', 'PX_VOLUME', 'TOT_RETURN_INDEX_GROSS_DVDS', 'month', 'ret']

Sample:


,ticker,date,CUR_MKT_CAP,EQY_SH_OUT,PX_HIGH,PX_LAST,PX_LOW,PX_OPEN,PX_VOLUME,TOT_RETURN_INDEX_GROSS_DVDS,month,ret
0,0966088D LN Equity,2014-03-28,NaN,NaN,NaN,180.0,NaN,NaN,NaN,180.0,2014-03,NaN
1,0966088D LN Equity,2014-03-31,3010.5744,NaN,NaN,180.0,NaN,NaN,NaN,180.0,2014-03,0.0
2,0966088D LN Equity,2014-04-01,3057.0408,NaN,NaN,180.0,NaN,NaN,NaN,180.0,2014-04,0.0
3,0966088D LN Equity,2014-04-02,3055.7887,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014-04,NaN
4,0966088D LN Equity,2014-04-03,3074.0135,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014-04,NaN


In [2]:
# Ensure date is datetime
train['date'] = pd.to_datetime(train['date'])

# Compute monthly returns from Total Return Index
# For each ticker, take the last available TR value per month
monthly = (train.dropna(subset=['TOT_RETURN_INDEX_GROSS_DVDS'])
           .sort_values(['ticker', 'date'])
           .groupby(['ticker', 'month'])
           .last()
           .reset_index())

# Compute monthly return from TR index
monthly = monthly.sort_values(['ticker', 'month']).reset_index(drop=True)
monthly['monthly_ret'] = monthly.groupby('ticker')['TOT_RETURN_INDEX_GROSS_DVDS'].pct_change()

print(f"Monthly observations: {len(monthly):,}")
print(f"Tickers: {monthly['ticker'].nunique()}")
print(f"Month range: {monthly['month'].min()} to {monthly['month'].max()}")
print(f"\nMonthly return stats:")
print(monthly['monthly_ret'].describe())

Monthly observations: 79,823
Tickers: 635
Month range: 2009-07 to 2023-12

Monthly return stats:
count    79188.000000
mean         0.010438
std          0.136268
min         -0.926075
25%         -0.037107
50%          0.007776
75%          0.054314
max         23.267497
Name: monthly_ret, dtype: float64


In [3]:
# Momentum signal: cumulative return from t-13 to t-2 (skip most recent month)
monthly = monthly.sort_values(['ticker', 'month']).reset_index(drop=True)

# Create lagged TR index values
monthly['tr_lag2'] = monthly.groupby('ticker')['TOT_RETURN_INDEX_GROSS_DVDS'].shift(1)
monthly['tr_lag13'] = monthly.groupby('ticker')['TOT_RETURN_INDEX_GROSS_DVDS'].shift(12)

# Momentum = (TR at t-2) / (TR at t-13) - 1
monthly['momentum_12_1'] = monthly['tr_lag2'] / monthly['tr_lag13'] - 1

valid_mom = monthly['momentum_12_1'].notna().sum()
print(f"=== SIGNAL 1: 12-MONTH SKIP-1 MOMENTUM ===")
print(f"Valid observations: {valid_mom:,}")
print(f"\nMomentum signal stats:")
print(monthly['momentum_12_1'].describe())
print(f"\nFirst valid month: {monthly.loc[monthly['momentum_12_1'].notna(), 'month'].min()}")

=== SIGNAL 1: 12-MONTH SKIP-1 MOMENTUM ===
Valid observations: 72,245

Momentum signal stats:
count    72245.000000
mean         0.108635
std          0.517042
min         -0.985709
25%         -0.097842
50%          0.074730
75%          0.264031
max         32.227008
Name: momentum_12_1, dtype: float64

First valid month: 2010-07


In [4]:
# Reversal signal: prior 1-month total return
# Already computed as 'monthly_ret' — that IS the 1-month return
# For the signal, we use the PRIOR month's return (shift by 1)

monthly['reversal_1m'] = monthly.groupby('ticker')['monthly_ret'].shift(1)

valid_rev = monthly['reversal_1m'].notna().sum()
print(f"=== SIGNAL 2: 1-MONTH SHORT-TERM REVERSAL ===")
print(f"Valid observations: {valid_rev:,}")
print(f"\nReversal signal stats:")
print(monthly['reversal_1m'].describe())
print(f"\nFirst valid month: {monthly.loc[monthly['reversal_1m'].notna(), 'month'].min()}")

=== SIGNAL 2: 1-MONTH SHORT-TERM REVERSAL ===
Valid observations: 78,553

Reversal signal stats:
count    78553.000000
mean         0.010108
std          0.136018
min         -0.926075
25%         -0.037476
50%          0.007577
75%          0.053920
max         23.267497
Name: reversal_1m, dtype: float64

First valid month: 2009-09


In [5]:
# Low-volatility signal: 90-day trailing realised volatility of daily returns
# Compute from daily data, then merge into the monthly frame

# Daily returns from TR index
train_sorted = train.sort_values(['ticker', 'date']).copy()
train_sorted['daily_ret'] = train_sorted.groupby('ticker')['TOT_RETURN_INDEX_GROSS_DVDS'].pct_change()

# 90-day rolling standard deviation (annualised)
train_sorted['vol_90d'] = (train_sorted.groupby('ticker')['daily_ret']
                           .transform(lambda x: x.rolling(90, min_periods=60).std())
                           * np.sqrt(252))

# Take the last vol reading per ticker per month
vol_monthly = (train_sorted.dropna(subset=['vol_90d'])
               .assign(month=train_sorted['date'].dt.to_period('M'))
               .sort_values(['ticker', 'date'])
               .groupby(['ticker', 'month'])['vol_90d']
               .last()
               .reset_index())

# Merge into the monthly frame
monthly = monthly.merge(vol_monthly, on=['ticker', 'month'], how='left')

valid_vol = monthly['vol_90d'].notna().sum()
print(f"=== SIGNAL 3: 90-DAY REALISED VOLATILITY ===")
print(f"Valid observations: {valid_vol:,}")
print(f"\nVolatility signal stats (annualised):")
print(monthly['vol_90d'].describe())
print(f"\nFirst valid month: {monthly.loc[monthly['vol_90d'].notna(), 'month'].min()}")

=== SIGNAL 3: 90-DAY REALISED VOLATILITY ===
Valid observations: 78,347

Volatility signal stats (annualised):
count    78347.000000
mean         0.321274
std          0.305143
min          0.000000
25%          0.198916
50%          0.274443
75%          0.379669
max         28.807548
Name: vol_90d, dtype: float64

First valid month: 2009-09


In [6]:
# Volume filter: is current 20-day average volume above the 60-day average?
# This signals "volume expansion" — confirms price moves have conviction

train_sorted = train_sorted.sort_values(['ticker', 'date'])

train_sorted['vol_ma20'] = (train_sorted.groupby('ticker')['PX_VOLUME']
                            .transform(lambda x: x.rolling(20, min_periods=15).mean()))
train_sorted['vol_ma60'] = (train_sorted.groupby('ticker')['PX_VOLUME']
                            .transform(lambda x: x.rolling(60, min_periods=40).mean()))

# Volume confirmation = 1 if 20-day avg > 60-day avg, else 0
train_sorted['vol_confirm'] = (train_sorted['vol_ma20'] > train_sorted['vol_ma60']).astype(int)

# Take last reading per ticker per month
vol_filter = (train_sorted.dropna(subset=['vol_ma60'])
              .assign(month=train_sorted['date'].dt.to_period('M'))
              .sort_values(['ticker', 'date'])
              .groupby(['ticker', 'month'])['vol_confirm']
              .last()
              .reset_index())

# Merge into monthly frame
monthly = monthly.merge(vol_filter, on=['ticker', 'month'], how='left')

valid_vf = monthly['vol_confirm'].notna().sum()
pct_confirmed = monthly['vol_confirm'].mean() * 100
print(f"=== VOLUME CONFIRMATION FILTER ===")
print(f"Valid observations: {valid_vf:,}")
print(f"Percent with volume confirmation: {pct_confirmed:.1f}%")
print(f"\nFirst valid month: {monthly.loc[monthly['vol_confirm'].notna(), 'month'].min()}")

=== VOLUME CONFIRMATION FILTER ===
Valid observations: 78,554
Percent with volume confirmation: 45.2%

First valid month: 2009-08


In [7]:
# Save the monthly signal frame
monthly.to_parquet('../data/working/monthly_signals.parquet')

# Summary of all signals
print("=== SIGNAL SUMMARY ===")
print(f"Total monthly observations: {len(monthly):,}")
print(f"Tickers: {monthly['ticker'].nunique()}")
print(f"Month range: {monthly['month'].min()} to {monthly['month'].max()}")
print(f"\nSignal coverage:")
print(f"  Momentum (12-1):     {monthly['momentum_12_1'].notna().sum():,} valid")
print(f"  Reversal (1m):       {monthly['reversal_1m'].notna().sum():,} valid")
print(f"  Volatility (90d):    {monthly['vol_90d'].notna().sum():,} valid")
print(f"  Volume filter:       {monthly['vol_confirm'].notna().sum():,} valid")
print(f"\nAll signals available from: 2010-07")
print(f"  (momentum constrains the start — needs 13 months of history)")

print(f"\nColumns in signal frame:")
print(list(monthly.columns))

=== SIGNAL SUMMARY ===
Total monthly observations: 79,823
Tickers: 635
Month range: 2009-07 to 2023-12

Signal coverage:
  Momentum (12-1):     72,245 valid
  Reversal (1m):       78,553 valid
  Volatility (90d):    78,347 valid
  Volume filter:       78,554 valid

All signals available from: 2010-07
  (momentum constrains the start — needs 13 months of history)

Columns in signal frame:
['ticker', 'month', 'date', 'CUR_MKT_CAP', 'EQY_SH_OUT', 'PX_HIGH', 'PX_LAST', 'PX_LOW', 'PX_OPEN', 'PX_VOLUME', 'TOT_RETURN_INDEX_GROSS_DVDS', 'ret', 'monthly_ret', 'tr_lag2', 'tr_lag13', 'momentum_12_1', 'reversal_1m', 'vol_90d', 'vol_confirm']


In [1]:
# === Diagnostic 3a: how were signals stored in NB04? ===
import os, json
os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())

with open('notebooks/04_signal_computation.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

print(f"NB04 has {len(nb['cells'])} cells.\n")

kw = ('def ', 'shift', 'momentum_12_1', 'vol_90d', 'reversal_1m',
      'monthly_ret', 'rolling', '.resample', 'pct_change', 'TOT_RETURN',
      'ffill', 'bfill', 'lag', 'cumprod')

for i, cell in enumerate(nb['cells']):
    if cell['cell_type'] != 'code':
        continue
    src = ''.join(cell['source'])
    hits = [k for k in kw if k in src]
    if len(hits) >= 2:
        print(f"\n{'='*78}\nCELL {i}  (hits: {hits})\n{'='*78}")
        print(src[:3500])
        if len(src) > 3500:
            print(f"... [+{len(src)-3500} more chars]")

NB04 has 8 cells.


CELL 1  (hits: ['monthly_ret', 'pct_change', 'TOT_RETURN'])
# Ensure date is datetime
train['date'] = pd.to_datetime(train['date'])

# Compute monthly returns from Total Return Index
# For each ticker, take the last available TR value per month
monthly = (train.dropna(subset=['TOT_RETURN_INDEX_GROSS_DVDS'])
           .sort_values(['ticker', 'date'])
           .groupby(['ticker', 'month'])
           .last()
           .reset_index())

# Compute monthly return from TR index
monthly = monthly.sort_values(['ticker', 'month']).reset_index(drop=True)
monthly['monthly_ret'] = monthly.groupby('ticker')['TOT_RETURN_INDEX_GROSS_DVDS'].pct_change()

print(f"Monthly observations: {len(monthly):,}")
print(f"Tickers: {monthly['ticker'].nunique()}")
print(f"Month range: {monthly['month'].min()} to {monthly['month'].max()}")
print(f"\nMonthly return stats:")
print(monthly['monthly_ret'].describe())

CELL 2  (hits: ['shift', 'momentum_12_1', 'TOT_RETURN', 'lag'])
# Momentum signa